# AKATSUKI Training — DeepSeek-R1-Distill-Qwen-7B

Menu → Runtime → Change runtime type → **A100 GPU** (Colab Pro)

---

In [ ]:
import torch
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {gpu.total_mem/1e9:.1f}GB | CUDA: {torch.version.cuda}")

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers accelerate bitsandbytes peft trl datasets scipy pyyaml
print('Dependencies installed')

In [ ]:
import os
if not os.path.exists('akatsuki'):
    !git clone https://github.com/kwkkd/akatsuki-hermes-integration akatsuki
%cd akatsuki

In [ ]:
from corpus_builder import build_corpus, build_dpo_corpus
build_corpus(num_articles=5000, num_dialogs=10000)
build_dpo_corpus(num_pairs=3000)
print('Corpus ready')

In [ ]:
!python continue_pretrain.py --model_id "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B" --qlora --batch_size 2 --lr 5e-5 --epochs 1

In [ ]:
!python train.py --model_id ./pretrained_akatsuki

In [ ]:
from inference import AkatsukiInference
from akatsuki_config import CONFIG
ai = AkatsukiInference(model_path=CONFIG.model.merged_path)
ai.load()

tests = ["너는 누구야?", "@Pain, C2 chain 구축해줘", "nmap SMB 취약점 스캔 명령어"]
for t in tests:
    print(f"=== {t} ===")
    print(ai.chat(t))
    print()

In [ ]:
!tar czf akatsuki_model.tar.gz merged_hacker_ai_model/
from google.colab import files
files.download('akatsuki_model.tar.gz')